# Run deseq2 on pseudobulk aggregated data

## 2025-10-23
### Palak Genge, High Resolution Translational Immunology, Allen Institute for Immunology
#### Objective: Run deseq2 analysis on pseudobulk aggregated counts for downstread fgsea analysis

In [47]:
# load all required packages
suppressPackageStartupMessages({
  library(data.table)  
  library(dplyr)       
  library(tidyr)       
  library(stringr)     
  library(ggplot2)     
  library(ggrepel)     
  library(grid)        
  library(DESeq2)      
  library(qvalue)      
  library(parallel)    
  library(tidyverse)
  library(purrr)
  library(fgsea)
})

In [48]:
source('../../00-utilities/functions/r/base-deseq2.R')

In [1]:
# Read combined counts and metadata made from pseudobulk
counts_matrix <- read.csv("../../../data/rna/pseudobulk/outputs/pseudo_bulk_matrices_raw_counts/combined_pseudobulk_counts.csv", row.names = 1)
colData <- read.csv("../../../data/rna/pseudobulk/outputs/pseudo_bulk_matrices_raw_counts/combined_pseudobulk_colData.csv", row.names = 1)

# make sure Healthy Plasma names match and replace spaces with dots in colData rownames (if counts_matrix has dots)
rownames(colData) <- gsub(" ", ".", rownames(colData))

# check for mismatch
mismatched <- setdiff(colnames(counts_matrix), rownames(colData))
if(length(mismatched) > 0){
  warning("Mismatched columns found in counts matrix: ", paste(mismatched, collapse = ", "))
}

# ensure columns match
common_cols <- intersect(colnames(counts_matrix), rownames(colData))
counts_matrix <- counts_matrix[, common_cols]
colData <- colData[common_cols, , drop = FALSE]

# create deseq2 object
dds <- DESeqDataSetFromMatrix(
  countData = counts_matrix,
  colData = colData,
  design = ~ cluster
)

# filter genes for low expression
dds <- dds[rowSums(counts(dds)) > 10, ]

# run deseq2
dds <- DESeq(dds)

# get tumor cluster labels only
clusters <- setdiff(unique(colData$cluster), "Healthy Plasma")

# creat output folder
dir.create("../../../data/rna/plasma/outputs/DEGs", showWarnings = FALSE)

# Loop over clusters and save csvs with regulation column
for (clust in clusters) {
  res <- results(dds, contrast = c("cluster", clust, "Healthy Plasma"))
  res <- res[order(res$padj), ]
  
  # add column to determine upregulation or down regulation
  res$regulation <- ifelse(
    is.na(res$padj), NA,
    ifelse(res$padj < 0.05 & res$log2FoldChange > 0, "up",
           ifelse(res$padj < 0.05 & res$log2FoldChange < 0, "down", "ns"))
  )
  
  # save csvs in folder with ndmm naming convetion
  out_file <- paste0("../../../data/rna/plasma/outputs/DEGs/DEGs_", clust, "_vs_Healthy.csv")
  write.csv(as.data.frame(res), out_file)
  message("✅ Saved DEGs for ", clust, " → ", out_file)
}

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    Filter, Find, Map, Position, Reduce, anyDuplicated, aperm, append,
    as.data.frame, basename, cbind, colnames, dirname, do.call,
    duplicated, eval, evalq, get, grep, grepl, intersect, is.unsorted,
    lapply, mapply, match, mget, order, paste, pmax, pmax.int, pmin,
    pmin.int, rank, rbind, rownames, sapply, saveRDS, setdiff, table,
    tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    I, expand.grid, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

